In [ ]:
import os
from urllib.parse import urlparse

import pandas as pd
from databricks.sdk.core import Config

from databricks import sql

config = Config()
connection = sql.connect(
    server_hostname=urlparse(config.host).hostname,
    http_path=f'/sql/1.0/warehouses/{os.environ["DATABRICKS_WAREHOUSE_ID"]}',
    credentials_provider=lambda: config.authenticate,
    catalog="demo_redshift", schema="gold", user_agent_entry="Cognition_Devin",
)


In [ ]:
query = """-- BI report: RFM segment sizes and value.
SELECT rfm_segment,
       COUNT(*)          AS customers,
       SUM(monetary)     AS total_monetary,
       CAST(AVG(frequency) AS BIGINT) AS avg_frequency
FROM   gold.rfm_segments
GROUP  BY rfm_segment
ORDER  BY rfm_segment;
"""
with connection.cursor() as cursor:
    cursor.execute(query)
    rfm_segments = pd.DataFrame.from_records(cursor.fetchall(), columns=[c[0] for c in cursor.description])
rfm_segments.head()


In [ ]:
query = """-- BI report: retention share per cohort x month offset.
SELECT cohort_month,
       month_offset,
       cohort_size,
       active_buyers,
       ROUND(100.0 * active_buyers / cohort_size, 2) AS retention_pct
FROM   gold.cohort_retention
ORDER  BY cohort_month, month_offset;
"""
with connection.cursor() as cursor:
    cursor.execute(query)
    cohort_retention = pd.DataFrame.from_records(cursor.fetchall(), columns=[c[0] for c in cursor.description])
cohort_retention.head()


In [ ]:
connection.close()
